In [1]:
import argparse
import re
import sys

from utils import (
    wildcard_match,
    read_intensities,
    read_unique_kmer_positions,
    parse_snv_string,
    normalize_snv_region,
    get_snv_aligned_wildcards,
    match_snv_aligned_kmers,
    run_aff_regression,
    sample_rand_kmers_per_allele,
    run_rand_regression_from_region_map,
    print_motif_effect_table,
    reverse_complement,
)

genome =  "../../../../../d/OneDrive - McGill University/repos/hg38_ucsc.fa"
intensities = read_intensities("/mnt/c/Users/Aki/Desktop/EUBAR/data/intensities/DNase_GABPA_MCF7_probeIntensity.bed")
kmers = read_unique_kmer_positions("/mnt/c/Users/Aki/Desktop/EUBAR/data/array/MCF7_Array_8mer_DNase.txt")
dhs = 1
no_covariates = True
# region = "chr5:1295105-1295140"
# region = "chr6:1295220-1295228"
num_random = 1000
mode = "neg-binomial"

kmer_size = 8
# snv_list =  "chr5:1295113:G>A"
snv_str =  "chr5:1295113:C>T"
# snv_list =  "chr5:1295113:C>T,chr5:1295135:C>T"

/home/aki/miniconda3/lib/python3.8/site-packages/scipy/__init__.py:146: UserWarning: A NumPy version >=1.16.5 and <1.23.0 is required for this version of SciPy (detected version 1.24.4
  warnings.warn(f"A NumPy version >={np_minversion} and <{np_maxversion}"


In [2]:
chrom, snv_pos, ref, alt = parse_snv_string(snv_str)
snv_info = normalize_snv_region(
        chrom, snv_pos, ref, alt, genome, kmer_size
    )
snv_info["snv_str"] = snv_str
snv_info["wildcards"] = dict(
    get_snv_aligned_wildcards(snv_info, kmer_size)
)

allele_region_offsets, wildcards, matched_regions = match_snv_aligned_kmers(
    snv_info, kmers, kmer_size
)

In [3]:
# Step 1: build allele_matched_kmers
allele_matched_kmers = {}
for motif_pos, wildcard in wildcards:
    snv_index = wildcard.index(".")
    allele_matched_kmers.setdefault(motif_pos, {}).setdefault(snv_index, {})
    for kmer in kmers:
        if wildcard_match(kmer, wildcard):
            allele = kmer[snv_index]
            for region_id in kmers[kmer]:
                allele_matched_kmers[motif_pos][snv_index].setdefault(
                    allele, []
                ).append((wildcard, kmer, region_id))

In [13]:
import pandas as pd

# ----------------------------
# AFF: build hits + LM inputs
# ----------------------------

def build_hit_table(wildcards, kmers, allele_region_offsets, kmer_size):
    """
    Returns a DataFrame of:
      motif_pos, snv_index, allele, region,
      kmer (kmer from file), rc_kmer (revcomp),
      is_reverse, wildcard, wildcard_pos
    """
    rows = []
    for motif_pos, wildcard in wildcards:
        j = wildcard.index(".")  # wildcard index in forward coords

        for kmer in kmers.keys():
            # forward match
            if wildcard_match(kmer, wildcard):
                allele = kmer[j]
                rc = reverse_complement(kmer)
                for region in kmers[kmer].keys():
                    wildcard_pos = (
                        allele_region_offsets.get(motif_pos, {})
                                          .get(allele, {})
                                          .get(region, None)
                    )
                    rows.append(dict(
                        motif_pos=motif_pos,
                        snv_index=j,
                        allele=allele,
                        region=region,
                        kmer=kmer,
                        rc_kmer=rc,
                        is_reverse=False,
                        wildcard=wildcard,
                        wildcard_pos=wildcard_pos,
                    ))

            # reverse-complement match (kmer in file is forward, but it matches via its revcomp)
            rc = reverse_complement(kmer)
            if wildcard_match(rc, wildcard):
                allele = rc[j]  # allele in forward coordinates
                for region in kmers[kmer].keys():
                    wildcard_pos = (
                        allele_region_offsets.get(motif_pos, {})
                                          .get(allele, {})
                                          .get(region, None)
                    )
                    rows.append(dict(
                        motif_pos=motif_pos,
                        snv_index=j,
                        allele=allele,
                        region=region,
                        kmer=kmer,      # original key in the kmer file
                        rc_kmer=rc,     # the string that actually matched the wildcard
                        is_reverse=True,
                        wildcard=wildcard,
                        wildcard_pos=wildcard_pos,
                    ))

    df = pd.DataFrame(rows)

    # keep only rows that actually ended up in your offsets map (i.e., used downstream)
    df = df[df["wildcard_pos"].notna()].copy()

    # de-dup in case the same (motif_pos, allele, region) is hit multiple ways
    df.drop_duplicates(subset=["motif_pos", "snv_index", "allele", "region"], inplace=True)

    return df


def region_length(region):
    _, coords = region.split(":")
    start, end = map(int, coords.split("-"))
    return end - start


def add_lm_columns(hits, intensities, fold_half=True):
    """
    Adds:
      length, y (raw), y_round,
      lp_raw, lp (optionally folded 0-0.5),
      sl
    """
    df = hits.copy()
    df["length"] = df["region"].apply(region_length)

    # response
    df["y"] = df["region"].map(intensities)  # float
    df["y_round"] = df["y"].apply(lambda x: int(round(x)) if pd.notna(x) else None)

    # covariates
    df["lp_raw"] = df["wildcard_pos"] / df["length"]
    if fold_half:
        df["lp"] = df["lp_raw"].apply(lambda x: 1 - x if x > 0.5 else x)
    else:
        df["lp"] = df["lp_raw"]

    df["sl"] = df["length"]
    return df


def add_allele_dummies(df):
    out = df.copy()
    for base in ["A", "C", "G", "T"]:
        out[f"allele_{base}"] = (out["allele"] == base).astype(int)
    return out


def build_aff_debug_df(wildcards, kmers, allele_region_offsets, intensities, kmer_size, fold_half=True):
    """
    Convenience: hits -> LM columns -> allele dummies
    """
    hits = build_hit_table(wildcards, kmers, allele_region_offsets, kmer_size)
    df = add_lm_columns(hits, intensities, fold_half=fold_half)
    df = add_allele_dummies(df)
    return df


In [14]:
# ----------------------------
# RAND: build LM inputs table
# ----------------------------

def region_length(region):
    _, coords = region.split(":")
    start, end = map(int, coords.split("-"))
    return end - start


def fold_lp_half(x):
    return 1 - x if x > 0.5 else x


def build_rand_debug_df(region_sets, intensities, kmer_size, motif_pos, fold_half=True):
    """
    region_sets: dict allele -> {region: offset}
      (output of sample_rand_kmers_per_allele)

    Builds a DataFrame with:
      region, allele (the allele owning that region), offset,
      wildcard_pos (= offset + motif_pos), length, lp_raw, lp, sl,
      y, y_round, and allele dummy columns.
    """
    # union of regions across alleles
    all_regions = set()
    for allele, d in region_sets.items():
        all_regions |= set(d.keys())

    rows = []
    for region in sorted(all_regions):
        L = region_length(region)

        # find the allele + offset for this region
        offset = None
        owner = None
        for allele, d in region_sets.items():
            if region in d:
                offset = d[region]
                owner = allele
                break

        wildcard_pos = (int(offset) + int(motif_pos)) if offset is not None else None
        lp_raw = (wildcard_pos / L) if (wildcard_pos is not None and L > 0) else None
        lp = fold_lp_half(lp_raw) if (fold_half and lp_raw is not None) else lp_raw

        y = intensities.get(region)

        rows.append(dict(
            motif_pos=motif_pos,
            region=region,
            allele=owner,
            offset=offset,
            wildcard_pos=wildcard_pos,
            length=L,
            lp_raw=lp_raw,
            lp=lp,
            sl=L,
            y=y,
            y_round=(int(round(y)) if y is not None else None),
            y_missing=(y is None),
        ))

    df = pd.DataFrame(rows)

    # allele dummy columns
    for base in ["A", "C", "G", "T"]:
        df[f"allele_{base}"] = (df["allele"] == base).astype(int)

    return df


In [18]:
# ----------------------------
# RUN IT (AFF + RAND)
# ----------------------------

# AFF debug table (Perl-style folding by default)
debug_aff = build_aff_debug_df(
    wildcards=wildcards,
    kmers=kmers,
    allele_region_offsets=allele_region_offsets,
    intensities=intensities,
    kmer_size=kmer_size,
    fold_half=True,          # set False for raw 0–1
)

# take a peek
cols_aff = [
    "motif_pos","snv_index","allele","region","is_reverse",
    "wildcard_pos","length","lp_raw","lp","sl","y_round",
    "wildcard","kmer","rc_kmer"
]
display(debug_aff[cols_aff].head(30))

# optional: save
# debug_aff.to_csv("aff_lm_inputs_debug.tsv", sep="\t", index=False)


# RAND debug table
# 1) sample random regions first (uses your existing variables)
rand_regions_per_allele, used_regions = sample_rand_kmers_per_allele(
    kmers=kmers,
    matched_regions=matched_regions,
    snv_str=snv_str,
    kmer_size=kmer_size,
    rand_n=num_random
)

# 2) pick a motif position (0..kmer_size-1)
motif_pos_for_rand = 3

# 3) pull the allele->(region->offset) map for that motif position
region_sets = rand_regions_per_allele[motif_pos_for_rand]

# 4) build the debug table
debug_rand = build_rand_debug_df(
    region_sets=region_sets,
    intensities=intensities,
    kmer_size=kmer_size,
    motif_pos=motif_pos_for_rand,
    fold_half=True
)

display(debug_rand.head(30))

# optional: save
# debug_rand.to_csv(f"rand_lm_inputs_debug_motifpos{motif_pos_for_rand}.tsv", sep="\t", index=False)


,motif_pos,snv_index,allele,region,is_reverse,wildcard_pos,length,lp_raw,lp,sl,y_round,wildcard,kmer,rc_kmer
0,0,7,T,chr1:995900-996116,True,176.0,216,0.814815,0.185185,216,4,CAGCCCC.,AGGGGCTG,CAGCCCCT
1,0,7,T,chr1:3796480-3796680,True,160.0,200,0.800000,0.200000,200,90,CAGCCCC.,AGGGGCTG,CAGCCCCT
2,0,7,T,chr1:4826200-4826380,True,23.0,180,0.127778,0.127778,180,2,CAGCCCC.,AGGGGCTG,CAGCCCCT
3,0,7,T,chr1:5161460-5161640,True,116.0,180,0.644444,0.355556,180,1,CAGCCCC.,AGGGGCTG,CAGCCCCT
4,0,7,T,chr1:5637300-5637480,True,60.0,180,0.333333,0.333333,180,2,CAGCCCC.,AGGGGCTG,CAGCCCCT
5,0,7,T,chr1:6058660-6058840,True,141.0,180,0.783333,0.216667,180,2,CAGCCCC.,AGGGGCTG,CAGCCCCT
6,0,7,T,chr1:6483120-6483320,True,117.0,200,0.585000,0.415000,200,2,CAGCCCC.,AGGGGCTG,CAGCCCCT
7,0,7,T,chr1:7954100-7954360,True,28.0,260,0.107692,0.107692,260,67,CAGCCCC.,AGGGGCTG,CAGCCCCT
8,0,7,T,chr1:9627080-9627300,True,63.0,220,0.286364,0.286364,220,7,CAGCCCC.,AGGGGCTG,CAGCCCCT
9,0,7,T,chr1:12161920-12162100,True,102.0,180,0.566667,0.433333,180,4,CAGCCCC.,AGGGGCTG,CAGCCCCT


,motif_pos,region,allele,offset,wildcard_pos,length,lp_raw,lp,sl,y,y_round,y_missing,allele_A,allele_C,allele_G,allele_T
0,3,chr10:116427780-116427960,A,107,110,180,0.611111,0.388889,180,2.10019,2,False,1,0,0,0
1,3,chr10:121554965-121555109,A,80,83,144,0.576389,0.423611,144,1.22773,1,False,1,0,0,0
2,3,chr10:20858400-20858580,G,86,89,180,0.494444,0.494444,180,0.58462,1,False,0,0,1,0
3,3,chr10:25060100-25060320,G,162,165,220,0.750000,0.250000,220,2.87005,3,False,0,0,1,0
4,3,chr10:27709400-27709600,G,125,128,200,0.640000,0.360000,200,2.15995,2,False,0,0,1,0
5,3,chr11:23932640-23932729,T,2,5,89,0.056180,0.056180,89,0.52514,1,False,0,0,0,1
6,3,chr11:39306820-39307040,C,79,82,220,0.372727,0.372727,220,2.65531,3,False,0,1,0,0
7,3,chr11:47179620-47179800,C,70,73,180,0.405556,0.405556,180,1.69563,2,False,0,1,0,0
8,3,chr11:66892260-66892460,C,37,40,200,0.200000,0.200000,200,2.89400,3,False,0,1,0,0
9,3,chr11:92707220-92707460,A,170,173,240,0.720833,0.279167,240,0.52514,1,False,1,0,0,0


In [6]:
results_aff = []
for motif_pos, snv_dict in allele_matched_kmers.items():
    for snv_index in snv_dict:
        rows = run_aff_regression(
            motif_pos=motif_pos,
            snv_index=snv_index,
            allele_region_offsets=allele_region_offsets,
            allele_matched_kmers=allele_matched_kmers,
            intensities=intensities,
            region_seq=snv_info["region_seq"],
            model_type=mode,
            include_covariates=not no_covariates,
        )
        results_aff.extend(rows)


In [7]:
results_aff

[{'wildcard_kmer': 'CAGCCCC.',
  'filled_kmer': 'CAGCCCCA',
  'motif_pos': 0,
  'snp_index': 7,
  'type': 'AFF',
  'allele': 'A',
  'coef': -0.1564417688623111,
  'pval': 6.054237131879554e-12,
  'absolute_pos': 7},
 {'wildcard_kmer': 'CAGCCCC.',
  'filled_kmer': 'CAGCCCCG',
  'motif_pos': 0,
  'snp_index': 7,
  'type': 'AFF',
  'allele': 'G',
  'coef': 0.19414422166972647,
  'pval': 2.423156902087232e-14,
  'absolute_pos': 7},
 {'wildcard_kmer': 'CAGCCCC.',
  'filled_kmer': 'CAGCCCCT',
  'motif_pos': 0,
  'snp_index': 7,
  'type': 'AFF',
  'allele': 'T',
  'coef': -0.09016303727588262,
  'pval': 0.0002176000777654326,
  'absolute_pos': 7},
 {'wildcard_kmer': 'CAGCCCC.',
  'filled_kmer': 'CAGCCCCC',
  'motif_pos': 0,
  'snp_index': 7,
  'type': 'AFF',
  'allele': 'C',
  'coef': 'NA',
  'pval': 'NA',
  'absolute_pos': 7},
 {'wildcard_kmer': 'AGCCCC.T',
  'filled_kmer': 'AGCCCCAT',
  'motif_pos': 1,
  'snp_index': 6,
  'type': 'AFF',
  'allele': 'A',
  'coef': -0.09281206866951186,
  'pv